In [167]:
# Import dependencies 
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib

In [168]:
# Load the dataset
df = pd.read_csv('../data/bank_cleaned.csv')

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,prev_contacted
0,58,management,married,tertiary,no,2143,yes,no,missing,5,may,261,1,-1,0,missing,no,0
1,44,technician,single,secondary,no,29,yes,no,missing,5,may,151,1,-1,0,missing,no,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,missing,5,may,76,1,-1,0,missing,no,0
3,35,management,married,tertiary,no,231,yes,no,missing,5,may,139,1,-1,0,missing,no,0
4,28,management,single,tertiary,no,447,yes,yes,missing,5,may,217,1,-1,0,missing,no,0


In [169]:
# Encode target column
df['y'] = df['y'].map({'no': 0, 'yes': 1})

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43190 entries, 0 to 43189
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             43190 non-null  int64
 1   job             43190 non-null  str  
 2   marital         43190 non-null  str  
 3   education       43190 non-null  str  
 4   default         43190 non-null  str  
 5   balance         43190 non-null  int64
 6   housing         43190 non-null  str  
 7   loan            43190 non-null  str  
 8   contact         43190 non-null  str  
 9   day             43190 non-null  int64
 10  month           43190 non-null  str  
 11  duration        43190 non-null  int64
 12  campaign        43190 non-null  int64
 13  pdays           43190 non-null  int64
 14  previous        43190 non-null  int64
 15  poutcome        43190 non-null  str  
 16  y               43190 non-null  int64
 17  prev_contacted  43190 non-null  int64
dtypes: int64(9), str(9)
memory usage: 8.0

Based on our findings in EDA, we are going to create new features.

In [170]:
df['job_student_or_retired_feat'] = df['job'].isin(['student', 'retired']).astype(int)

df.drop(columns = ['job'], inplace = True)

df.groupby('job_student_or_retired_feat')['y'].mean()

job_student_or_retired_feat
0    0.107003
1    0.243836
Name: y, dtype: float64

In [171]:
df['marital_single_feat'] = df['marital'].isin(['single']).astype(int)

df.drop(columns = ['marital'], inplace = True)

df.groupby('marital_single_feat')['y'].mean()

marital_single_feat
0    0.103355
1    0.148948
Name: y, dtype: float64

In [172]:
df['education_tertiary_feat'] = df['education'].isin(['tertiary']).astype(int)

df.drop(columns = ['education'], inplace = True)

df.groupby('education_tertiary_feat')['y'].mean()

education_tertiary_feat
0    0.101210
1    0.150204
Name: y, dtype: float64

In [173]:
df['contact_cellular_telephone_feat'] = df['contact'].isin(['cellular', 'telephone']).astype(int)

df.drop(columns = ['contact'], inplace = True)

df.groupby('contact_cellular_telephone_feat')['y'].mean()

contact_cellular_telephone_feat
0    0.041351
1    0.146028
Name: y, dtype: float64

In [174]:
df['month_dec_oct_sep_mar_feat'] = df['month'].isin(['dec', 'oct', 'sep', 'mar']).astype(int)

df.drop(columns = ['month'], inplace = True)

df.groupby('month_dec_oct_sep_mar_feat')['y'].mean()

month_dec_oct_sep_mar_feat
0    0.100169
1    0.471658
Name: y, dtype: float64

In [175]:
df['poutcome_success_feat'] = df['poutcome'].isin(['success']).astype(int)

df.drop(columns = ['poutcome'], inplace = True)

df.groupby('poutcome_success_feat')['y'].mean()

poutcome_success_feat
0    0.098262
1    0.643961
Name: y, dtype: float64

In [176]:
# Encode yes and no in housing, loan, and default
df['housing_feat'] = df['housing'].map({'yes': 1, 'no': 0})
df.drop(columns = ['housing'], inplace = True)

df['loan_feat'] = df['loan'].map({'yes': 1, 'no': 0})
df.drop(columns = ['loan'], inplace = True)

df['default_feat'] = df['default'].map({'yes': 1, 'no': 0})
df.drop(columns = ['default'], inplace = True)

# Check the datatype of our dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43190 entries, 0 to 43189
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   age                              43190 non-null  int64
 1   balance                          43190 non-null  int64
 2   day                              43190 non-null  int64
 3   duration                         43190 non-null  int64
 4   campaign                         43190 non-null  int64
 5   pdays                            43190 non-null  int64
 6   previous                         43190 non-null  int64
 7   y                                43190 non-null  int64
 8   prev_contacted                   43190 non-null  int64
 9   job_student_or_retired_feat      43190 non-null  int64
 10  marital_single_feat              43190 non-null  int64
 11  education_tertiary_feat          43190 non-null  int64
 12  contact_cellular_telephone_feat  43190 non-null  int64
 1

In [177]:
df.head()

,age,balance,day,duration,campaign,pdays,previous,y,prev_contacted,job_student_or_retired_feat,marital_single_feat,education_tertiary_feat,contact_cellular_telephone_feat,month_dec_oct_sep_mar_feat,poutcome_success_feat,housing_feat,loan_feat,default_feat
0,58,2143,5,261,1,-1,0,0,0,0,0,1,0,0,0,1,0,0
1,44,29,5,151,1,-1,0,0,0,0,1,0,0,0,0,1,0,0
2,33,2,5,76,1,-1,0,0,0,0,0,0,0,0,0,1,1,0
3,35,231,5,139,1,-1,0,0,0,0,0,1,0,0,0,1,0,0
4,28,447,5,217,1,-1,0,0,0,0,1,1,0,0,0,1,1,0


Feature engineering on num cols

In [178]:
df['duration_bin'] = pd.cut(df['duration'],
                                 bins = [0, 103, 180, 318, np.inf],
                                 labels = ['Q1', 'Q2', 'Q3', 'Q4'])

df['duration_bin_Q3_Q4_feat'] = df['duration_bin'].isin(['Q3', 'Q4']).astype(int)

df.drop(columns = ['duration_bin'], inplace = True)

df.groupby('duration_bin_Q3_Q4_feat')['y'].mean()

duration_bin_Q3_Q4_feat
0    0.031152
1    0.201680
Name: y, dtype: float64

In [179]:
df['age_bin_feat'] = pd.cut(df['age'],
                       bins = [18, 33, 39, 48, 95],
                       labels = ['Q1', 'Q2', 'Q3', 'Q4'])

df.groupby('age_bin_feat')['y'].mean()

age_bin_feat
Q1    0.134877
Q2    0.102644
Q3    0.089524
Q4    0.132330
Name: y, dtype: float64

Due to its insignificance, we are going to remove both age and age bin

In [180]:
df.drop(columns = ['age', 'age_bin_feat'], inplace = True)

In [181]:
df['previous_any_feat'] = (df['previous'] > 0).astype(int)

df.groupby('previous_any_feat')['y'].mean()

previous_any_feat
0    0.091332
1    0.227376
Name: y, dtype: float64

In [182]:
df['day_bin_feat'] = pd.cut(df['day'],
                       bins = [1, 8, 16, 21, 31],
                       labels = ['Q1', 'Q2', 'Q3', 'Q4'])

df.groupby('day_bin_feat')['y'].mean()

day_bin_feat
Q1    0.118789
Q2    0.136108
Q3    0.085839
Q4    0.117681
Name: y, dtype: float64

In [183]:
# Also the same for day because it does not have any effect
df.drop(columns = ['day', 'day_bin_feat'], inplace = True)

In [184]:
df['campaign_bin_feat'] = (df['campaign'] > 3).astype(int)

df.drop(columns = ['campaign'], inplace = True)

df.groupby('campaign_bin_feat')['y'].mean()

campaign_bin_feat
0    0.127842
1    0.073641
Name: y, dtype: float64

In [185]:
# Non pdays are set to -1, we are going to change that to 999 for a better corr
df['pdays_replaced_feat'] = df['pdays'].replace(-1, 999)

In [186]:
# Log balance column and duration column for their skew
df['log_balance_feat'] = np.log(df['balance'] - df['balance'].min() + 1)
df['log_duration_feat'] = np.log(df['duration'] - df['duration'].min() + 1)

In [187]:
# Create the features columns 
features = df.drop(columns = ['y']).columns.tolist()

In [188]:
# Get their asb corr with y
corr = df.corr()['y'].drop('y').abs().sort_values(ascending = False)

corr.head(15)

duration                           0.397383
log_duration_feat                  0.342248
poutcome_success_feat              0.303997
duration_bin_Q3_Q4_feat            0.266009
month_dec_oct_sep_mar_feat         0.235882
pdays_replaced_feat                0.175730
prev_contacted                     0.164182
previous_any_feat                  0.164182
contact_cellular_telephone_feat    0.147334
housing_feat                       0.138300
job_student_or_retired_feat        0.107182
pdays                              0.101437
previous                           0.091759
education_tertiary_feat            0.070508
campaign_bin_feat                  0.069328
Name: y, dtype: float64

In [189]:
# Select corrs above 0.1 for our candidates
selected = corr[corr > 0.1].index.tolist()

X_selected = df[selected]

X_selected

,duration,log_duration_feat,poutcome_success_feat,duration_bin_Q3_Q4_feat,month_dec_oct_sep_mar_feat,pdays_replaced_feat,prev_contacted,previous_any_feat,contact_cellular_telephone_feat,housing_feat,job_student_or_retired_feat,pdays
0,261,5.564520,0,1,0,999,0,0,0,1,0,-1
1,151,5.017280,0,0,0,999,0,0,0,1,0,-1
2,76,4.330733,0,0,0,999,0,0,0,1,0,-1
3,139,4.934474,0,0,0,999,0,0,0,1,0,-1
4,217,5.379897,0,1,0,999,0,0,0,1,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...
43185,977,6.884487,0,1,0,999,0,0,1,0,0,-1
43186,456,6.122493,0,1,0,999,0,0,1,0,1,-1
43187,1127,7.027315,1,1,0,184,1,1,1,0,1,184
43188,508,6.230481,0,1,0,999,0,0,1,0,0,-1


In [190]:
# Now check for corr among the selected features
# Find pairs with abs corr above 0.7
# Remove the one in the pair that has a lower corr with y
corr_matrix = X_selected.corr().abs()

features_to_drop = set()

target_corr_selected = corr[selected]

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if corr_matrix.iloc[i, j] > 0.7:
            feat_i = corr_matrix.columns[i]
            feat_j = corr_matrix.columns[j]

            if target_corr_selected[feat_i] >= target_corr_selected[feat_j]:
                features_to_drop.add(feat_j)
            else:
                features_to_drop.add(feat_i)

final_features = [f for f in selected if f not in features_to_drop]

df[final_features].head()

,duration,poutcome_success_feat,month_dec_oct_sep_mar_feat,pdays_replaced_feat,contact_cellular_telephone_feat,housing_feat,job_student_or_retired_feat
0,261,0,0,999,0,1,0
1,151,0,0,999,0,1,0
2,76,0,0,999,0,1,0
3,139,0,0,999,0,1,0
4,217,0,0,999,0,1,0


In [191]:
# Create the new dataset and add y to it
final_df = df[final_features]

final_df['y'] = df['y'].values

final_df.head()

,duration,poutcome_success_feat,month_dec_oct_sep_mar_feat,pdays_replaced_feat,contact_cellular_telephone_feat,housing_feat,job_student_or_retired_feat,y
0,261,0,0,999,0,1,0,0
1,151,0,0,999,0,1,0,0
2,76,0,0,999,0,1,0,0
3,139,0,0,999,0,1,0,0
4,217,0,0,999,0,1,0,0


In [192]:
# Check their corr with y again to test the logic of what we did
corrf = final_df.corr()['y'].drop('y').abs().sort_values(ascending = False)

corrf

duration                           0.397383
poutcome_success_feat              0.303997
month_dec_oct_sep_mar_feat         0.235882
pdays_replaced_feat                0.175730
contact_cellular_telephone_feat    0.147334
housing_feat                       0.138300
job_student_or_retired_feat        0.107182
Name: y, dtype: float64

In [193]:
# Check the whole dataset corr for logic test
final_df.corr()

,duration,poutcome_success_feat,month_dec_oct_sep_mar_feat,pdays_replaced_feat,contact_cellular_telephone_feat,housing_feat,job_student_or_retired_feat,y
duration,1.000000,0.041320,0.020113,-0.004399,0.013274,0.004031,0.019616,0.397383
poutcome_success_feat,0.041320,1.000000,0.183708,-0.422081,0.113255,-0.091184,0.075261,0.303997
month_dec_oct_sep_mar_feat,0.020113,0.183708,1.000000,-0.167641,0.108406,-0.140729,0.133926,0.235882
pdays_replaced_feat,-0.004399,-0.422081,-0.167641,1.000000,-0.288405,-0.040888,-0.048083,-0.175730
contact_cellular_telephone_feat,0.013274,0.113255,0.108406,-0.288405,1.000000,-0.210930,0.048970,0.147334
housing_feat,0.004031,-0.091184,-0.140729,-0.040888,-0.210930,1.000000,-0.172916,-0.138300
job_student_or_retired_feat,0.019616,0.075261,0.133926,-0.048083,0.048970,-0.172916,1.000000,0.107182
y,0.397383,0.303997,0.235882,-0.175730,0.147334,-0.138300,0.107182,1.000000


In [194]:
continuos_cols = ['duration', 'pdays_replaced_feat']

scaler = RobustScaler()

final_df[continuos_cols] = scaler.fit_transform(final_df[continuos_cols])

final_df.head()

,duration,poutcome_success_feat,month_dec_oct_sep_mar_feat,pdays_replaced_feat,contact_cellular_telephone_feat,housing_feat,job_student_or_retired_feat,y
0,0.376744,0,0,0.0,0,1,0,0
1,-0.134884,0,0,0.0,0,1,0,0
2,-0.483721,0,0,0.0,0,1,0,0
3,-0.190698,0,0,0.0,0,1,0,0
4,0.172093,0,0,0.0,0,1,0,0


In [195]:
# Save the new dataset
final_df.to_csv('../data/bank_final.csv', index = False)

# Save the scaler model
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']